# Практика 8. Групування і зведені таблиці: `groupby`, агрегувальні функції, `pivot_table`, `crosstab`

**Варіант 1 — Київ.** Середньорічна температура: 9.5 °C; сезонна амплітуда: 14 °C.

Мета: побудувати довгий (tidy) набір даних за 2021–2024 роки та застосувати до нього `groupby`, `.agg()`, `pivot_table()` і `crosstab()`.

## Підготовка довгого набору

Кожен рядок містить одне спостереження: місто, рік, місяць і температуру. Такий формат зручний для групування й подальшого аналізу.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
variant = 1
city = 'Київ'
base_temp = 9.5
amplitude = 14
years = [2021, 2022, 2023, 2024]

rows = []
for year in years:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        rows.append({
            'місто': city,
            'рік': year,
            'місяць': month,
            'температура': round(base_temp + seasonal + noise, 1),
        })

climate = pd.DataFrame(rows)
print(f'Варіант {variant}: {city}')
print('Форма набору:', climate.shape)
display(climate.head(8))
assert climate.shape == (48, 4)
assert climate[['місто', 'рік', 'місяць', 'температура']].notna().all().all()

Варіант 1: Київ
Форма набору: (48, 4)


,місто,рік,місяць,температура
0,Київ,2021,1,-4.0
1,Київ,2021,2,-2.8
2,Київ,2021,3,3.1
3,Київ,2021,4,11.0
4,Київ,2021,5,16.3
5,Київ,2021,6,21.4
6,Київ,2021,7,25.1
7,Київ,2021,8,22.4


## Завдання 1. `groupby` і `.agg()` за роками

Застосуємо split-apply-combine: `groupby` розділяє рядки за роком, `.agg()` обчислює кілька статистик для кожної групи, а pandas об'єднує результати в одну таблицю.

In [ ]:
year_summary = climate.groupby('рік')['температура'].agg(['mean', 'min', 'max'])
year_summary.columns = ['середня', 'мінімум', 'максимум']
display(year_summary)

**Висновок за роками.** Середні значення треба порівнювати послідовно від 2021 до 2024 року. Через випадковий шум у моделі вони можуть трохи коливатися в різні боки, тому чотири спостереження не дають достатніх підстав називати це стійким потеплінням. Тут коректніше говорити про невеликі міжрічні коливання навколо заданої середньорічної температури.

## Завдання 2. `groupby` і `.agg()` за місяцями

Тепер групуємо 4 спостереження кожного місяця й одночасно рахуємо середню температуру та стандартне відхилення між роками.

In [ ]:
month_summary = climate.groupby('місяць')['температура'].agg(['mean', 'std'])
month_summary.columns = ['середня', 'std']
display(month_summary)
most_variable_month = month_summary['std'].idxmax()
print(f'Найбільший розкид між роками має місяць: {most_variable_month}')
print('Припущення: перехідні сезони можуть бути нестабільнішими через швидкі зміни погоди.')

**Висновок за місяцями.** Місяць із найбільшим `std` є найнестабільнішим у цій синтетичній вибірці. Це може бути випадковим наслідком шуму, але метеорологічно правдоподібне пояснення — перехідний сезон: навесні та восени температура змінюється швидше, ніж у стабільніші періоди зими або літа.

## Завдання 3. `pivot_table()`

Зробимо таблицю, де місяці є індексом, роки — стовпцями, а значення — середня температура. `pivot_table()` явно вказує агрегування, тому здатна працювати навіть із дубльованими комбінаціями ключів.

In [ ]:
temperature_pivot = climate.pivot_table(
    index='місяць',
    columns='рік',
    values='температура',
    aggfunc='mean'
)
display(temperature_pivot.round(2))

**Порівняння форматів.** Зведена таблиця зручніша для швидкого читання людиною: легко порівняти один місяць між роками. Довгий формат зручніший для `groupby`, фільтрації, додавання нових вимірів і побудови графіків. `pivot_table()` — це зручне представлення результату, а не заміна tidy-структури для всіх операцій.

## Завдання 4. Похідні категорії та `crosstab()`

Додамо сезон і прапорець, що показує, чи температура вища за базову середньорічну температуру варіанта.

In [ ]:
season_map = {
    12: 'зима', 1: 'зима', 2: 'зима',
    3: 'весна', 4: 'весна', 5: 'весна',
    6: 'літо', 7: 'літо', 8: 'літо',
    9: 'осінь', 10: 'осінь', 11: 'осінь'
}
climate['сезон'] = climate['місяць'].map(season_map)
climate['тепліше_за_середнє'] = climate['температура'] > base_temp

season_order = ['зима', 'весна', 'літо', 'осінь']
season_crosstab = pd.crosstab(
    climate['сезон'],
    climate['тепліше_за_середнє']
).reindex(season_order)
display(season_crosstab)

by_groupby = climate.groupby(['сезон', 'тепліше_за_середнє']).size().unstack(fill_value=0)
assert season_crosstab.fillna(0).equals(by_groupby.reindex(season_order, fill_value=0))
print('Crosstab узгоджується з еквівалентним groupby(...).size().')

**Висновок за сезонами.** Улітку більшість або всі значення мають бути в категорії `True`, бо сезонна складова досягає піку в липні. Однак шум може змістити окреме спостереження. `crosstab()` показує таблицю частот комбінацій категорій, а `groupby(...).size()` дає той самий підрахунок у більш загальній формі.

## Завдання 5. `pivot()` на власному наборі

У вихідному наборі кожна пара `(місяць, рік)` має рівно один рядок, тому `pivot()` працює без агрегування. Далі додамо другий вимірювальний пункт і покажемо, чому з'являється помилка дублікатів.

In [ ]:
temperature_pivot_direct = climate.pivot(
    index='місяць', columns='рік', values='температура'
)
display(temperature_pivot_direct.head())
print('У вихідному climate:', climate.groupby(['місяць', 'рік']).size().unique(), 'рядок на кожну пару.')

two_stations = pd.concat([
    climate.assign(станція='центр'),
    climate.assign(станція='північ')
], ignore_index=True)

try:
    two_stations.pivot(index='місяць', columns='рік', values='температура')
except ValueError as error:
    print('Після додавання станції pivot() завершується помилкою:')
    print(error)
else:
    raise AssertionError('Очікувалася помилка через дублікати ключів.')

**Пояснення.** Після додавання двох станцій кожна пара `(місяць, рік)` має вже два значення температури. `pivot()` не має правила, як вибрати одне з них, тому виникає `ValueError`. `pivot_table()` у цій ситуації працював би, якщо додати `станція` до індексу або стовпців, чи явно вказати агрегування, наприклад `aggfunc='mean'`.

## Контрольні питання

1. **Split-apply-combine:** `groupby()` спочатку split-ить дані на групи за ключем, потім apply-ить до кожної групи функцію (`mean`, `min`, `max`, `std`), а combine об'єднує результати в Series або DataFrame.

2. **`pivot()` і `pivot_table()`:** `pivot()` вимагає унікальної комбінації індексу та стовпця. `pivot_table()` може агрегувати кілька значень в одну клітинку, тому не ламається на дублікати, якщо задано `aggfunc`.

3. **`crosstab()`:** це зручна таблиця частот для двох або більше категоріальних змінних. Еквівалент через `groupby(...).size()` гнучкіший, але часто потребує `unstack(fill_value=0)`, щоб отримати такий самий табличний вигляд.

4. **Чому tidy-формат:** у довгому форматі одна змінна має один стовпець, а одне спостереження — один рядок. Це дозволяє однаково групувати, фільтрувати й додавати нові виміри. Формат із роками чи місяцями в назвах стовпців менш гнучкий і перед аналізом часто потребує `melt()`.

## Підсумок

На довгому наборі з 48 кліматичних спостережень застосовано `groupby` з кількома агрегатами, `pivot_table` для зручного читання, `crosstab` для частот категорій і `pivot` для випадку з унікальними ключами. Додавання другого вимірювального пункту показало, чому дублікати ключів змушують використовувати агрегування.